# Pedalboard + Surge XT Fundamentals

This notebook is a lab manual for the renderer layer of PatchPilot / PPv2: **parameters + MIDI in, audio out**.

The goal is not to memorize Pedalboard calls. The goal is to build a mental model of what Python is doing when it pretends to be a tiny DAW host:

1. create or read an audio buffer,
2. measure it,
3. process it with effects,
4. load a synth plugin,
5. send MIDI into the synth,
6. render audio,
7. save audio and metadata.

We will keep the path narrow on purpose. No JUCE, Xcode, CoreML, PyTorch, embeddings, FAISS, real-time app work, or full PPv2 architecture appears here. Those topics make more sense after the renderer layer feels ordinary.

## 0. What this notebook is and is not

By the end, you should be able to:

- represent audio as NumPy arrays,
- explain sample rate, frames, channels, RMS, peak, and dBFS,
- use Pedalboard built-in effects,
- locate and load Surge XT as a plugin,
- send MIDI notes to Surge XT,
- inspect and set plugin parameters,
- render audio reproducibly enough for small dataset experiments,
- save WAV files and metadata.

This is not an ML notebook. It is the lab bench under the ML work.

## 1. Environment check

Before we touch audio, we need to know which Python is running the notebook. Audio plugin hosting is unusually sensitive to environment details: one wrong interpreter can mean Pedalboard is installed in one place while Jupyter is using another.

In [ ]:
import os
import sys
import platform
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
(ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from pedalboard import (
    Pedalboard,
    Gain,
    Reverb,
    Compressor,
    Limiter,
    LowpassFilter,
    HighpassFilter,
    load_plugin,
)
from pedalboard.io import AudioFile
from mido import Message

from pedalboard_surge_utils import (
    SAMPLE_RATE,
    SILENCE_THRESHOLD_DBFS,
    assert_render_is_sane,
    dbfs,
    discover_surge_candidates,
    duration_seconds,
    ensure_channels_first,
    ensure_output_dirs,
    likely_continuous_parameters,
    load_surge,
    mean_absolute_difference,
    package_version,
    parameter_table,
    peak,
    play_audio,
    plot_spectrum,
    plot_waveform,
    plugin_metadata,
    print_parameter_matches,
    read_audio,
    render_note,
    rms,
    search_parameters,
    set_param_if_exists,
    set_raw_param_if_exists,
    spectral_centroid,
    summarize_audio,
    write_audio,
    write_parameter_table_csv,
)

try:
    import pandas as pd
except ImportError as exc:
    pd = None
    print(f"Optional pandas import failed: {type(exc).__name__}: {exc}")

try:
    import librosa
except ImportError as exc:
    librosa = None
    print(f"Optional librosa import failed: {type(exc).__name__}: {exc}")

OUTPUT_DIRS = ensure_output_dirs(ROOT)

print(f"Project root: {ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
for package_name in ["pedalboard", "numpy", "matplotlib", "mido", "jupyter", "ipykernel", "pandas", "librosa"]:
    print(f"{package_name}: {package_version(package_name)}")

if sys.version_info[:2] != (3, 12):
    print("Note: this tutorial can run here, but PPv2's long-term target should stay on Python 3.12.")

### Checkpoint

The line that matters most is `Python executable`. If it is not inside `tonarpy` / `~/.venvs/audio`, stop and switch kernels before continuing. Debugging plugin hosts through the wrong interpreter is a very efficient way to waste an afternoon.

## 2. Audio as arrays

Pedalboard ultimately processes blocks of samples. Before Surge appears, we need an audio buffer that we completely understand. A sine wave is boring in the useful way: if anything changes, we know the code did it.

In [ ]:
sample_rate = SAMPLE_RATE
duration = 1.0
frequency_hz = 220.0
amplitude = 0.25

frame_count = int(sample_rate * duration)
time = np.arange(frame_count) / sample_rate
sine_mono = amplitude * np.sin(2 * np.pi * frequency_hz * time)

print(f"sample_rate={sample_rate}")
print(f"duration={duration}s")
print(f"frame_count={frame_count}")
print(f"mono shape={sine_mono.shape}")
print(f"min={sine_mono.min():.3f}, max={sine_mono.max():.3f}")

A mono NumPy vector is enough for a first signal, but it is not enough for a full notebook. Pedalboard file I/O commonly returns audio in **channel-first** shape: `(channels, samples)`. Generated NumPy examples often start as `(samples,)`. If we keep switching shapes by hand, every section becomes a little shape bug waiting to happen.

That is the first real pain point, so now the helper `ensure_channels_first` earns its keep: it turns mono vectors and common stereo layouts into one stable convention.

In [ ]:
sine = ensure_channels_first(sine_mono)
sine_stereo = np.vstack([sine[0], sine[0]])

print(f"channel-first mono shape: {sine.shape}")
print(f"channel-first stereo shape: {sine_stereo.shape}")
assert sine.shape == (1, frame_count)
assert sine_stereo.shape == (2, frame_count)

The next repeated pain point is measurement. Listening matters, but it is not enough for a renderer that will later create datasets. We need quick, boring measurements that catch silence, clipping, and duration mistakes.

The helpers `rms`, `dbfs`, `peak`, `duration_seconds`, `plot_waveform`, `plot_spectrum`, and `play_audio` exist so each later render can be checked in the same way.

In [ ]:
summary = summarize_audio(sine_stereo, sample_rate, label="220 Hz sine")
summary

In [ ]:
plot_waveform(sine_stereo, sample_rate, "220 Hz sine wave", seconds=0.03)
plot_spectrum(sine_stereo, sample_rate, "220 Hz sine spectrum")
play_audio(sine_stereo, sample_rate)

### Practice checkpoint

Try this yourself before opening the solution.

**Task:** Create a 440 Hz sine wave at half the amplitude of the current one, convert it to channel-first shape, and assert that its peak is close to `0.125`.

<details>
<summary>Show solution</summary>

```python
practice_frequency = 440.0
practice_amplitude = amplitude / 2
practice = practice_amplitude * np.sin(2 * np.pi * practice_frequency * time)
practice = ensure_channels_first(practice)
assert np.isclose(peak(practice), 0.125, atol=1e-3)
```

</details>

In [ ]:
# Your turn: fill this in, then run the assertion.
practice_frequency = 440.0
practice_amplitude = amplitude / 2
practice = practice_amplitude * np.sin(2 * np.pi * practice_frequency * time)
practice = ensure_channels_first(practice)
assert np.isclose(peak(practice), 0.125, atol=1e-3)
print("Practice check passed.")

## 3. First Pedalboard chain with built-in effects

Now we have a known buffer and known measurements. That makes it safe to introduce the central Pedalboard idea: an effects chain is a list of processors that transforms an audio buffer into another audio buffer.

We are deliberately using generated audio first. Surge adds plugin discovery, MIDI, and parameter state. None of that helps us learn what an effects chain is.

In [ ]:
board = Pedalboard([
    Gain(gain_db=-6),
    LowpassFilter(cutoff_frequency_hz=2000),
    Reverb(room_size=0.25),
    Limiter(threshold_db=-1),
])

processed_sine = ensure_channels_first(board(sine_stereo, sample_rate, reset=True))

before_after = [
    summarize_audio(sine_stereo, sample_rate, label="dry sine"),
    summarize_audio(processed_sine, sample_rate, label="effected sine"),
]

if pd is not None:
    display(pd.DataFrame(before_after))
else:
    for row in before_after:
        print(row)

In [ ]:
plot_waveform(sine_stereo, sample_rate, "Dry sine", seconds=0.03)
plot_waveform(processed_sine, sample_rate, "After Gain + Lowpass + Reverb + Limiter", seconds=0.03)
plot_spectrum(sine_stereo, sample_rate, "Dry sine spectrum")
plot_spectrum(processed_sine, sample_rate, "Effected sine spectrum")
play_audio(processed_sine, sample_rate)

### Comprehension checkpoint

`Pedalboard([...])` is useful here because every item in the list is an **effect**: it takes audio in and returns audio out. A synth instrument is different. An instrument takes MIDI in and creates audio, so we will render the instrument first and then put effects after the render.

## 4. AudioFile read/write

A renderer becomes useful when its output can leave memory. For dataset generation, that means two things: audio files and metadata rows. We start with only the audio file.

In [ ]:
test_sine_path = OUTPUT_DIRS["renders"] / "test_sine.wav"
write_audio(test_sine_path, sine_stereo, sample_rate)

read_back, read_sample_rate = read_audio(test_sine_path)
read_summary = summarize_audio(read_back, read_sample_rate, label="read-back sine")

print(f"Wrote: {test_sine_path}")
print(read_summary)

assert read_sample_rate == sample_rate
assert read_back.shape[0] == 2
assert abs(read_back.shape[1] - frame_count) <= 1
assert rms(read_back) > 0

### Practice checkpoint

**Task:** Save the effected sine as `outputs/renders/effected_sine.wav`, read it back, and verify that the read-back RMS is nonzero.

<details>
<summary>Show solution</summary>

```python
effected_path = OUTPUT_DIRS["renders"] / "effected_sine.wav"
write_audio(effected_path, processed_sine, sample_rate)
effected_back, effected_rate = read_audio(effected_path)
assert effected_rate == sample_rate
assert rms(effected_back) > 0
```

</details>

In [ ]:
effected_path = OUTPUT_DIRS["renders"] / "effected_sine.wav"
write_audio(effected_path, processed_sine, sample_rate)
effected_back, effected_rate = read_audio(effected_path)
assert effected_rate == sample_rate
assert rms(effected_back) > 0
print("Saved and verified", effected_path)

## 5. Locate and load Surge XT

Now the problem changes. We do not just need an effect; we need a plugin bundle on disk, and we need the **instrument** version of Surge XT. The helper `find_surge_plugin` / `load_surge` exists because hard-coding one path is brittle across machines, user installs, VST3 vs AU installs, and multi-plugin containers.

The search order is:

1. `SURGE_PLUGIN_PATH`, if set,
2. system VST3,
3. user VST3,
4. system Audio Unit,
5. user Audio Unit.

VST3 is preferred when available.

In [ ]:
candidates = discover_surge_candidates(scan_plugin_names=True)
for candidate in candidates:
    print(f"path={candidate.path}")
    print(f"  format={candidate.plugin_format}")
    print(f"  names={candidate.plugin_names or 'not available'}")
    print(f"  selected={candidate.selected_plugin_name}")
    if candidate.scan_error:
        print(f"  scan_error={candidate.scan_error}")

surge = load_surge(initialization_timeout=30.0)
metadata = plugin_metadata(surge)
metadata

In [ ]:
assert metadata["is_instrument"] is True
print("Loaded an instrument plugin. MIDI rendering is allowed.")

## 6. Parameter introspection

A plugin is not just a black box. It exposes parameters, but parameter names are plugin-defined and may change across versions. That is why we inspect before setting.

Two forms matter:

- **natural value**: often meaningful to humans, such as Hz, dB, milliseconds, or a named mode,
- **`raw_value`**: always normalized on `[0, 1]`, useful for generic storage and dataset metadata.

Pedalboard guesses natural ranges where it can. Treat those guesses as helpful, not sacred.

In [ ]:
param_rows = parameter_table(surge)
param_csv = OUTPUT_DIRS["analysis"] / "surge_parameters.csv"
write_parameter_table_csv(param_rows, param_csv)
print(f"Saved {len(param_rows)} parameters to {param_csv}")

if pd is not None:
    display(pd.DataFrame(param_rows).head(20))
else:
    for row in param_rows[:20]:
        print(row)

The next helpers solve a concrete annoyance: Surge has many parameters, and we should not pretend we know the exact keys before looking. `search_parameters` returns structured matches; `print_parameter_matches` gives a fast readable view; `set_param_if_exists` and `set_raw_param_if_exists` let experiments skip missing keys without lying about success.

In [ ]:
for terms in [("osc",), ("filter",), ("cutoff",), ("res",), ("attack",), ("release",), ("volume",), ("level",)]:
    print("\n---", terms)
    print_parameter_matches(surge, terms, max_results=10)

### Practice checkpoint

**Task:** Search for parameters related to envelope sustain. Read the matches and choose the key that looks most plausible on your installed Surge XT.

<details>
<summary>Show solution pattern</summary>

```python
sustain_matches = search_parameters(surge, "sustain")
for row in sustain_matches[:10]:
    print(row["key"], row["value"], row["raw_value"], row["display_name"])
```

</details>

In [ ]:
sustain_matches = search_parameters(surge, "sustain")
for row in sustain_matches[:10]:
    print(row["key"], row["value"], row["raw_value"], row["display_name"])
print(f"Found {len(sustain_matches)} sustain-like parameters.")

## 7. First MIDI render from Surge

We can finally do the thing this repo is for: send MIDI into Surge and receive audio back.

A `Message("note_on", ..., time=0)` starts at the beginning of the render. A matching `note_off` with `time=2.0` ends the note two seconds into the rendered buffer. Pedalboard returns `duration` seconds of audio.

In [ ]:
messages = [
    Message("note_on", note=60, velocity=100, time=0),
    Message("note_off", note=60, velocity=0, time=2.0),
]

surge_audio = ensure_channels_first(
    surge(
        messages,
        duration=3.0,
        sample_rate=sample_rate,
        num_channels=2,
        buffer_size=8192,
        reset=True,
    )
)

surge_summary = assert_render_is_sane(surge_audio, sample_rate, expected_duration=3.0)
surge_summary

In [ ]:
surge_path = OUTPUT_DIRS["renders"] / "surge_c4_default.wav"
write_audio(surge_path, surge_audio, sample_rate)
plot_waveform(surge_audio, sample_rate, "Surge C4 default render", seconds=0.08)
plot_spectrum(surge_audio, sample_rate, "Surge C4 default spectrum")
play_audio(surge_audio, sample_rate)
print(f"Saved {surge_path}")

### Practice checkpoint

Try this yourself before opening the solution.

**Task:** Render MIDI note 72 instead of 60 and save it as `surge_c5.wav`. Add an assertion that the render is louder than `-60 dBFS`.

<details>
<summary>Show solution</summary>

```python
surge_c5 = render_note(surge, note=72, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=True)
assert dbfs(surge_c5) > -60
write_audio(OUTPUT_DIRS["renders"] / "surge_c5.wav", surge_c5, sample_rate)
```

</details>

In [ ]:
surge_c5 = render_note(surge, note=72, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=True)
assert dbfs(surge_c5) > SILENCE_THRESHOLD_DBFS
write_audio(OUTPUT_DIRS["renders"] / "surge_c5.wav", surge_c5, sample_rate)
print("C5 practice render passed.")

## 8. Making parameter changes safely

The concrete problem now is not "how do I set a filter?" It is "how do I set a parameter without depending on a fragile name I have not verified?"

So the workflow is:

1. search likely terms,
2. inspect matches,
3. choose a key,
4. save the original value,
5. set the parameter explicitly,
6. render,
7. measure whether audio changed,
8. restore the original value when the experiment is done.

In [ ]:
search_groups = [
    ("osc",), ("type",), ("saw",), ("shape",),
    ("filter",), ("cutoff",), ("resonance",), ("res",),
    ("amp",), ("attack",), ("release",), ("sustain",),
    ("scene",), ("volume",), ("level",),
]

for terms in search_groups:
    matches = search_parameters(surge, *terms)
    if matches:
        print(f"\n{terms}: {len(matches)} matches")
        for row in matches[:5]:
            print(f"  {row['key']}: value={row['value']!r}, raw={row['raw_value']!r}, name={row['display_name']!r}")

In [ ]:
cutoff_candidates = likely_continuous_parameters(surge, ["cutoff"])
resonance_candidates = likely_continuous_parameters(surge, ["resonance", "res"])
filter_candidates = likely_continuous_parameters(surge, ["filter"])
fallback_candidates = likely_continuous_parameters(surge, ["volume", "level", "attack", "release", "osc"])

experiment_key = None
for key_list in [cutoff_candidates, resonance_candidates, filter_candidates, fallback_candidates]:
    if key_list:
        experiment_key = key_list[0]
        break

print("cutoff candidates:", cutoff_candidates[:8])
print("resonance candidates:", resonance_candidates[:8])
print("filter candidates:", filter_candidates[:8])
print("selected experiment_key:", experiment_key)


Natural-unit assignment is pleasant when the value means something to a human. Raw assignment is better for generic dataset metadata because every plugin parameter can be represented on `[0, 1]`.

The next cell demonstrates both, but only after a real parameter key has been discovered.

In [ ]:
if experiment_key is None:
    print("No suitable continuous parameter found; skipping parameter assignment demo.")
else:
    parameter = surge.parameters[experiment_key]
    original_value = getattr(parameter, "value", None)
    original_raw = parameter.raw_value
    print(f"Using {experiment_key}")
    print(f"  original natural value: {original_value!r}")
    print(f"  original raw value: {original_raw!r}")

    if isinstance(original_value, (int, float)):
        set_param_if_exists(surge, experiment_key, original_value)
        print("  natural-unit assignment round trip succeeded")
    else:
        print("  natural value is not numeric; skipping natural-unit assignment")

    set_raw_param_if_exists(surge, experiment_key, 0.25)
    print(f"  raw value after setting low: {surge.parameters[experiment_key].raw_value:.3f}")
    set_raw_param_if_exists(surge, experiment_key, float(original_raw))
    print("  restored original raw value")

## 9. Render comparison experiment

A parameter demo does not count unless the audio changes measurably. Listening is useful, but here we will also calculate RMS, peak, spectral centroid, and mean absolute sample difference from the default render.

In [ ]:
variant_rows = []
variant_audio = {}

if experiment_key is None:
    print("No continuous parameter available; skipping comparison experiment.")
else:
    original_raw = float(surge.parameters[experiment_key].raw_value)
    raw_values = [0.2, 0.5, 0.8]
    for raw_value in raw_values:
        set_raw_param_if_exists(surge, experiment_key, raw_value)
        audio = render_note(surge, note=60, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=True)
        label = f"{experiment_key}_raw_{raw_value:.2f}"
        path = OUTPUT_DIRS["renders"] / f"surge_c4_{label}.wav"
        write_audio(path, audio, sample_rate)
        variant_audio[label] = audio
        variant_rows.append({
            "label": label,
            "parameter_key": experiment_key,
            "raw_value": raw_value,
            "path": str(path),
            "rms": rms(audio),
            "dbfs": dbfs(audio),
            "peak": peak(audio),
            "spectral_centroid_hz": spectral_centroid(audio, sample_rate),
            "mean_abs_difference_from_default": mean_absolute_difference(surge_audio, audio),
        })
    set_raw_param_if_exists(surge, experiment_key, original_raw)

if variant_rows and pd is not None:
    display(pd.DataFrame(variant_rows))
elif variant_rows:
    for row in variant_rows:
        print(row)
else:
    print("No variants were rendered.")

In [ ]:
if variant_rows:
    max_difference = max(row["mean_abs_difference_from_default"] for row in variant_rows)
    assert max_difference > 1e-5, "Parameter changed, but audio did not change measurably."
    for label, audio in variant_audio.items():
        print(label, summarize_audio(audio, sample_rate, label=label))
        plot_spectrum(audio, sample_rate, f"Spectrum: {label}")
        play_audio(audio, sample_rate)
else:
    print("No variants were rendered.")

### Practice checkpoint

**Task:** Change the raw values list to five values and add a metadata column named `duration_seconds`. Your validation should still require at least one measurable audio difference.

<details>
<summary>Show solution sketch</summary>

```python
raw_values = np.linspace(0.1, 0.9, 5)
# Inside the loop:
row["duration_seconds"] = duration_seconds(audio, sample_rate)
assert max(row["mean_abs_difference_from_default"] for row in rows) > 1e-5
```

</details>

## 10. Built-in effects after Surge render

Now the earlier Pedalboard chain returns, but in the correct place. Surge turns MIDI into an audio buffer. Effects process that audio buffer afterward.

That is why we do **not** put the instrument plugin itself inside the effects `Pedalboard([...])` chain for this workflow.

In [ ]:
fx = Pedalboard([
    Compressor(threshold_db=-24, ratio=2.5),
    Reverb(room_size=0.35),
    Limiter(threshold_db=-1),
])

processed_surge = ensure_channels_first(fx(surge_audio, sample_rate, reset=True))
processed_surge_path = OUTPUT_DIRS["renders"] / "surge_c4_default_with_fx.wav"
write_audio(processed_surge_path, processed_surge, sample_rate)

if pd is not None:
    display(pd.DataFrame([
        summarize_audio(surge_audio, sample_rate, label="dry Surge"),
        summarize_audio(processed_surge, sample_rate, label="Surge + effects"),
    ]))
else:
    print(summarize_audio(surge_audio, sample_rate, label="dry Surge"))
    print(summarize_audio(processed_surge, sample_rate, label="Surge + effects"))

plot_waveform(processed_surge, sample_rate, "Surge render through effects", seconds=0.08)
play_audio(processed_surge, sample_rate)
print(f"Saved {processed_surge_path}")

## 11. Reset, state, and repeatability

Plugin state can persist between renders. That is not a bug; it is how effects, envelopes, LFOs, delays, and internal plugin state work.

For independent dataset examples, `reset=True` asks Pedalboard to clear plugin state before each render. For streaming chunks of one continuous performance, `reset=False` is often correct because tails and envelopes should continue across chunk boundaries.

We will not demand bit-perfect determinism from a complex synth plugin. Instead, we measure whether repeated renders are identical or merely close.

In [ ]:
repeat_a = render_note(surge, note=60, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=True)
repeat_b = render_note(surge, note=60, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=True)
repeat_difference = mean_absolute_difference(repeat_a, repeat_b)
print(f"reset=True repeat mean absolute difference: {repeat_difference:.10f}")

stream_a = render_note(surge, note=60, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=True)
stream_b = render_note(surge, note=60, duration=3.0, note_length=2.0, sample_rate=sample_rate, reset=False)
stream_difference = mean_absolute_difference(stream_a, stream_b)
print(f"reset=False follow-up mean absolute difference: {stream_difference:.10f}")

if repeat_difference < 1e-7:
    print("Repeated reset renders are effectively identical on this setup.")
else:
    print("Repeated reset renders differ on this setup; treat repeatability as a measured property, not an assumption.")

## 12. Tiny dataset-generation preview, not ML

This is the PPv2 bridge in miniature: set explicit parameters, render fixed MIDI, reject silence, save WAV, append metadata.

We keep the loop tiny on purpose. The point is to learn the workflow, not to create a dataset yet.

In [ ]:
import csv

metadata_rows = []

dataset_key = experiment_key
if dataset_key is None:
    print("No suitable continuous parameter found; skipping tiny dataset preview.")
else:
    original_raw = float(surge.parameters[dataset_key].raw_value)
    if "cutoff" in dataset_key.lower() or "filter" in dataset_key.lower():
        raw_values = np.geomspace(0.05, 0.95, 12)
        sampling_note = "geometric raw sweep for cutoff/filter-like parameter"
    else:
        raw_values = np.linspace(0.05, 0.95, 12)
        sampling_note = "linear raw sweep for generic continuous parameter"
    print(sampling_note)

    for index, raw_value in enumerate(raw_values):
        set_raw_param_if_exists(surge, dataset_key, float(raw_value))
        audio = render_note(surge, note=60, duration=2.5, note_length=1.75, sample_rate=sample_rate, reset=True)
        level = dbfs(audio)
        if level <= SILENCE_THRESHOLD_DBFS:
            print(f"Skipping silent render at raw={raw_value:.3f}: {level:.2f} dBFS")
            continue
        filename = f"tiny_{index:03d}_{dataset_key}_raw_{raw_value:.3f}.wav".replace("/", "_")
        path = OUTPUT_DIRS["renders"] / filename
        write_audio(path, audio, sample_rate)
        metadata_rows.append({
            "index": index,
            "note": 60,
            "velocity": 100,
            "duration_seconds": duration_seconds(audio, sample_rate),
            "parameter_key": dataset_key,
            "raw_value": float(raw_value),
            "dbfs": level,
            "peak": peak(audio),
            "rms": rms(audio),
            "spectral_centroid_hz": spectral_centroid(audio, sample_rate),
            "path": str(path),
        })
    set_raw_param_if_exists(surge, dataset_key, original_raw)

metadata_path = OUTPUT_DIRS["analysis"] / "tiny_patch_metadata.csv"
if metadata_rows:
    with metadata_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=list(metadata_rows[0].keys()))
        writer.writeheader()
        writer.writerows(metadata_rows)
    print(f"Saved {len(metadata_rows)} metadata rows to {metadata_path}")
    if pd is not None:
        display(pd.DataFrame(metadata_rows))
else:
    print("No metadata rows created.")

### Practice checkpoint

**Task:** Modify the tiny loop to render three notes per parameter value: C3, C4, and C5. The metadata should include the MIDI note for each file.

<details>
<summary>Show solution sketch</summary>

```python
for raw_value in raw_values:
    for note in [48, 60, 72]:
        audio = render_note(surge, note=note, duration=2.5, note_length=1.75, sample_rate=sample_rate, reset=True)
        row["note"] = note
```

</details>

## 13. Troubleshooting

### `ImportError` from `load_plugin`

Check that the plugin path exists and that the notebook kernel is using the same Python environment where Pedalboard is installed.

### Plugin path not found

Set an explicit path before launching Jupyter:

```bash
export SURGE_PLUGIN_PATH="/absolute/path/to/Surge XT.vst3"
```

### Multi-plugin VST3 requires `plugin_name`

Use `VST3Plugin.get_plugin_names_for_file(path)` when available. The utilities scan plugin names and choose the Surge instrument-looking entry when possible.

### Loaded an effect instead of an instrument

MIDI rendering requires `plugin.is_instrument == True`. Effects process audio buffers; instruments create audio from MIDI.

### Silent render

Check MIDI note timing, velocity, duration, output channel count, and whether the patch volume/envelope/filter settings are muting the sound. The smoke test rejects renders below `-60 dBFS`.

### Parameter name mismatch

Do not hard-code a guessed Surge parameter key. Export `outputs/analysis/surge_parameters.csv`, search terms, inspect the matches, then set only verified keys.

### macOS plugin GUI, security, or quarantine issues

This notebook does not require opening the plugin GUI. If macOS blocks the plugin, open Surge XT once in a trusted host or clear quarantine only when you understand the source of the plugin.

### Notebook kernel not using `tonarpy`

Run the environment cell and inspect `sys.executable`. If it is wrong, install/register an ipykernel from the correct environment.

### `pip` installed packages into the wrong interpreter

Prefer:

```bash
python -m pip install package-name
```

from inside the activated environment, then confirm with `python -c "import sys; print(sys.executable)"`.

### Python 3.14 warning

This tutorial may run under Python 3.14 if the packages support it. PPv2 should still target Python 3.12 long-term unless you intentionally move the whole stack.

## 14. PPv2 bridge

This notebook teaches the renderer layer.

In PatchPilot / PPv2 terms:

1. explicit parameter settings become patch metadata,
2. MIDI messages define the performance probe,
3. Surge renders audio,
4. WAV files and metadata become a small dataset,
5. only then do spectrograms, embeddings, and ML targets make sense.

The important discipline is already visible here: every render should declare the parameters it depends on, measure the audio it produced, and save enough metadata that a future model can learn from the result instead of guessing what happened.